# FFHQ Phase Retrieval With DAPS on Colab

This notebook is a Colab-ready runner for the pixel-space FFHQ phase-retrieval experiments in this repo.

The default settings use the full current contents of `dataset/test-ffhq` and keep the same main hyperparameters as your earlier `4k_10x400` FFHQ phase-retrieval runs:

- `+data=test-ffhq`
- `+model=ffhq256ddpm`
- `+task=phase_retrieval`
- `+sampler=edm_daps`
- `task_group=pixel`
- `num_runs=4`
- `sampler.diffusion_scheduler_config.num_steps=10`
- `sampler.annealing_scheduler_config.num_steps=400`
- `batch_size=1` for safer Colab memory usage
- `data.start_id=0`, `data.end_id=len(dataset/test-ffhq)`

Before running, switch Colab to a GPU runtime with `Runtime -> Change runtime type -> GPU`.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import os
import subprocess

# Set one of these up before running this cell.
# Option 1: point REPO_URL at a GitHub copy of this repo.
# Option 2: set USE_DRIVE_REPO=True and point DRIVE_REPO_DIR at a Google Drive copy.
# Option 3: manually place the repo at /content/DAPS-main before running.
REPO_URL = "https://github.com/Seif-Hussein/daps_test.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/DAPS-main")
USE_DRIVE_REPO = False
DRIVE_REPO_DIR = "/content/drive/MyDrive/DAPS-main"

if USE_DRIVE_REPO:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path(DRIVE_REPO_DIR)
elif not REPO_DIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL, enable USE_DRIVE_REPO, or upload the repo to /content/DAPS-main first."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")
print(f"README exists: {(REPO_DIR / 'README.md').exists()}")

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab-pixel.txt"], check=True)

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import zipfile
import subprocess
from pathlib import Path

FFHQ_CHECKPOINT_ID = "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh"
FFHQ_TEST_SET_ID = "1IzbnLWPpuIw6Z2E4IKrRByh6ihDE5QLO"

checkpoints_dir = Path("checkpoints")
dataset_dir = Path("dataset")
checkpoints_dir.mkdir(exist_ok=True)
dataset_dir.mkdir(exist_ok=True)

ffhq_ckpt = checkpoints_dir / "ffhq256.pt"
if not ffhq_ckpt.exists():
    subprocess.run(
        [sys.executable, "-m", "gdown", f"https://drive.google.com/uc?id={FFHQ_CHECKPOINT_ID}", "-O", str(ffhq_ckpt)],
        check=True,
    )

test_ffhq_dir = dataset_dir / "test-ffhq"
test_ffhq_zip = dataset_dir / "test-ffhq.zip"
if not test_ffhq_dir.exists():
    subprocess.run(
        [sys.executable, "-m", "gdown", f"https://drive.google.com/uc?id={FFHQ_TEST_SET_ID}", "-O", str(test_ffhq_zip)],
        check=True,
    )
    with zipfile.ZipFile(test_ffhq_zip, "r") as zip_ref:
        zip_ref.extractall(dataset_dir)
    test_ffhq_zip.unlink()

num_test_images = len(list(test_ffhq_dir.glob("*.png")))
print("Checkpoint:", ffhq_ckpt)
print("Test images:", num_test_images)

In [ ]:
# Default settings run all files currently present in dataset/test-ffhq.
# batch_size stays at 1 to be friendlier to Colab GPU memory.
# For a quicker smoke test on Colab, lower NUM_RUNS and ANNEALING_STEPS.

from pathlib import Path

DATA_CONFIG = "test-ffhq"   # switch to "demo-ffhq" for a lighter run
MODEL_CONFIG = "ffhq256ddpm"
TASK_CONFIG = "phase_retrieval"
SAMPLER_CONFIG = "edm_daps"
TASK_GROUP = "pixel"

test_ffhq_files = sorted(Path("dataset/test-ffhq").glob("*.png"))
if not test_ffhq_files:
    raise ValueError("No PNG files found in dataset/test-ffhq")
num_test_ffhq_files = len(test_ffhq_files)

NUM_RUNS = 4
DIFFUSION_STEPS = 10
ANNEALING_STEPS = 400
BATCH_SIZE = 1
START_ID = 0
END_ID = num_test_ffhq_files
RUN_NAME = f"phase_retrieval_4k_10x400_testffhq_all{END_ID - START_ID}"
SAVE_DIR = "results/pixel/ffhq"
GPU = 0

print(f"Found {num_test_ffhq_files} test FFHQ files")
print(f"Running slice [{START_ID}:{END_ID}] with batch_size={BATCH_SIZE}")

# A smaller Colab-friendly profile, if you want it:
# NUM_RUNS = 1
# DIFFUSION_STEPS = 5
# ANNEALING_STEPS = 200
# BATCH_SIZE = 1
# START_ID = 0
# END_ID = 1
# RUN_NAME = "phase_retrieval_colab_smoke_test"

In [ ]:
import os
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "posterior_sample.py",
    f"+data={DATA_CONFIG}",
    f"+model={MODEL_CONFIG}",
    f"+task={TASK_CONFIG}",
    f"+sampler={SAMPLER_CONFIG}",
    f"task_group={TASK_GROUP}",
    f"save_dir={SAVE_DIR}",
    f"num_runs={NUM_RUNS}",
    f"sampler.diffusion_scheduler_config.num_steps={DIFFUSION_STEPS}",
    f"sampler.annealing_scheduler_config.num_steps={ANNEALING_STEPS}",
    f"batch_size={BATCH_SIZE}",
    f"data.start_id={START_ID}",
    f"data.end_id={END_ID}",
    f"name={RUN_NAME}",
    f"gpu={GPU}",
]

print(" \\\n".join(shlex.quote(part) for part in cmd))
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.run(cmd, check=True, env=env)

In [ ]:
import json
from pathlib import Path
from pprint import pprint

result_dir = Path(SAVE_DIR) / RUN_NAME
metrics = json.loads((result_dir / "metrics.json").read_text())
print("Result directory:", result_dir)
pprint(metrics)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

result_dir = Path(SAVE_DIR) / RUN_NAME
grid_path = result_dir / "grid_results.png"
sample_paths = sorted((result_dir / "samples").glob("*.png"))
traj_paths = sorted((result_dir / "trajectory").glob("*.png"))

display(Image.open(grid_path))

preview_paths = sample_paths[:min(4, len(sample_paths))]
if preview_paths:
    fig, axes = plt.subplots(1, len(preview_paths), figsize=(4 * len(preview_paths), 4))
    if len(preview_paths) == 1:
        axes = [axes]
    for ax, path in zip(axes, preview_paths):
        ax.imshow(Image.open(path))
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Samples:", len(sample_paths))
print("Trajectories:", len(traj_paths))

In [ ]:
import shutil
from pathlib import Path

result_dir = Path(SAVE_DIR) / RUN_NAME
archive_path = shutil.make_archive(str(result_dir), "zip", root_dir=result_dir)
print("Created:", archive_path)